In [1]:
import numpy as np

# ==========================================
# precision control
# ==========================================
DT = np.longdouble


def asDT(x):
    return np.asarray(x, dtype=DT)

# ==========================================
# MQMQA G^ex toy verifier (Eq. 17 structure)
# Δg uses ξ-form (Eq. 24-like):
#   Δg_r = g_r * [ a^p b^q / (a+b)^(p+q) ]
# where for k = X or Y:
#   a = Y_{A/k},  b = Y_{B/k}
# and Y_{m/k} is the 2-index "equivalent fraction" built from quadruplet fractions:
#   Y_{m/k} = sum_r X_r * ((δ_am+δ_bm)(δ_xk+δ_yk))/4
# ==========================================

# 2 cations (A,B), 2 anions (X,Y)
vars_ = ["AA_XX","AB_XX","BB_XX",
         "AA_XY","AB_XY","BB_XY",
         "AA_YY","AB_YY","BB_YY"]
idx = {v:i for i,v in enumerate(vars_)}
M = len(vars_)

# Parse each variable into (a,b,x,y)
def parse_name(name):
    cat, an = name.split("_")
    a,b = cat[0], cat[1]            # 'A''A', 'A''B', 'B''B'
    x,y = an[0], an[1]              # 'X''X', 'X''Y', 'Y''Y'
    return a,b,x,y

quad = [parse_name(v) for v in vars_]  # list of tuples (a,b,x,y)

def delta(u, v): return DT("1.0") if u == v else DT("0.0")

# weights w_p(m,k) for Y_{m/k} (vector length M)
def w_vec(m, k):
    w = np.zeros(M, dtype=DT)
    for p,(a,b,x,y) in enumerate(quad):
        w[p] = ((delta(a,m)+delta(b,m))*(delta(x,k)+delta(y,k))) / DT("4.0")
    return w

# Precompute weight vectors for Y_{A/X}, Y_{B/X}, Y_{A/Y}, Y_{B/Y}
W = {
    ("A","X"): w_vec("A","X"),
    ("B","X"): w_vec("B","X"),
    ("A","Y"): w_vec("A","Y"),
    ("B","Y"): w_vec("B","Y"),
}

def Y_and_derivs(n, m, k):
    """
    Y = (w·n)/N, N=sum(n).
    Y_p  = (w_p - Y)/N
    Y_pq = -(Y_p + Y_q)/N
    """
    n = asDT(n)
    N = np.sum(n, dtype=DT)
    w = W[(m,k)]
    A = np.dot(w, n)
    Y = A / N
    Y_p = (w - Y) / N
    # second derivative matrix
    Y_pq = np.zeros((M,M), dtype=DT)
    for p in range(M):
        for q in range(M):
            Y_pq[p,q] = -(Y_p[p] + Y_p[q]) / N
    return Y, Y_p, Y_pq

def delta_g_xi_and_derivs(n, p_exp=1.0, q_exp=1.0):
    """
    Δg only for kk = XX and YY quadruplets; /XY Δg = 0 (toy choice).
    For kk=XX use k='X'; for kk=YY use k='Y'.
    a=Y_{A/k}, b=Y_{B/k}. f = a^p b^q / (a+b)^(p+q).
    Use log-diff to get Δg_p and Δg_pq.
    """
    n = asDT(n)
    if np.any(n <= 0): raise ValueError("All n must be > 0.")

    dg    = np.zeros(M, dtype=DT)
    dg_p  = np.zeros((M,M), dtype=DT)
    dg_pq = np.zeros((M,M,M), dtype=DT)

    # toy coefficients per diagonal quadruplet
    g = {
        "AA_XX": DT("2000.0"), "AB_XX": DT("3000.0"), "BB_XX": DT("1500.0"),
        "AA_YY": DT("2500.0"), "AB_YY": DT("3500.0"), "BB_YY": DT("1200.0"),
    }

    # Precompute Y and derivatives for k=X and k=Y
    aX,aX_p,aX_pq = Y_and_derivs(n,"A","X")
    bX,bX_p,bX_pq = Y_and_derivs(n,"B","X")
    aY,aY_p,aY_pq = Y_and_derivs(n,"A","Y")
    bY,bY_p,bY_pq = Y_and_derivs(n,"B","Y")

    def fill(name, a, b, a_p, b_p, a_pq, b_pq):
        r = idx[name]
        gr = g[name]
        s   = a + b
        if a <= 0 or b <= 0 or s <= 0:
            raise ValueError("Need a,b,s positive for logs (choose n away from boundaries).")

        # f = a^p b^q / s^(p+q)
        r_exp = p_exp + q_exp
        f = (a**p_exp) * (b**q_exp) / (s**r_exp)
        val = gr * f
        dg[r] = val

        # log derivatives
        # ln f = p ln a + q ln b - r ln s
        s_p  = a_p + b_p
        s_pq = a_pq + b_pq

        Lambda_u = p_exp*(a_p/a) + q_exp*(b_p/b) - r_exp*(s_p/s)

        Lambda_uv = (
            p_exp*(a_pq/a - np.outer(a_p,a_p)/(a*a)) +
            q_exp*(b_pq/b - np.outer(b_p,b_p)/(b*b)) -
            r_exp*(s_pq/s - np.outer(s_p,s_p)/(s*s))
        )

        dg_p[r,:] = val * Lambda_u
        dg_pq[r,:,:] = val * (np.outer(Lambda_u, Lambda_u) + Lambda_uv)

    for nm in ["AA_XX","AB_XX","BB_XX"]:
        fill(nm, aX,bX, aX_p,bX_p, aX_pq,bX_pq)
    for nm in ["AA_YY","AB_YY","BB_YY"]:
        fill(nm, aY,bY, aY_p,bY_p, aY_pq,bY_pq)

    return dg, dg_p, dg_pq

def P_Q_and_derivs(n):
    """
    Eq.17-like prefactors with Z=1:
      P_{ij/XX} = 0.5*n_{ij/XY},  P_{ij/YY}=0.5*n_{ij/XY}
      Q_{AA/an} = 0.5*n_{AB/an},  Q_{BB/an}=0.5*n_{AB/an}   for an in {XX,XY,YY}
    """
    P = np.zeros(M, dtype=DT); Q = np.zeros(M, dtype=DT)
    Pp = np.zeros((M,M), dtype=DT); Qp = np.zeros((M,M), dtype=DT)

    # P terms (l=k): affect XX and YY, by cation pair ij
    for ij in ["AA","AB","BB"]:
        ixy = idx[f"{ij}_XY"]
        for kk in ["XX","YY"]:
            r = idx[f"{ij}_{kk}"]
            P[r] = DT("0.5") * n[ixy]
            Pp[r,ixy] = DT("0.5")

    # Q terms (j=i): AA and BB, for any anion pair
    for an in ["XX","XY","YY"]:
        iAB = idx[f"AB_{an}"]
        for ii in ["AA","BB"]:
            r = idx[f"{ii}_{an}"]
            Q[r] = DT("0.5") * n[iAB]
            Qp[r,iAB] = DT("0.5")

    return P,Q,Pp,Qp

def G_ex(n):
    n = asDT(n)
    dg,_,_ = delta_g_xi_and_derivs(n)
    P,Q,_,_ = P_Q_and_derivs(n)

    T1 = np.dot(n, dg)

    diag_l_eq_k = [idx[v] for v in ["AA_XX","AB_XX","BB_XX","AA_YY","AB_YY","BB_YY"]]
    T2 = np.dot(P[diag_l_eq_k], dg[diag_l_eq_k])

    diag_j_eq_i = [idx[v] for v in ["AA_XX","BB_XX","AA_XY","BB_XY","AA_YY","BB_YY"]]
    T3 = np.dot(Q[diag_j_eq_i], dg[diag_j_eq_i])

    return DT("0.5") * (T1 + T2 + T3)

def H_ex_analytic(n):
    n = asDT(n)
    dg, dg_p, dg_pq = delta_g_xi_and_derivs(n)
    P,Q,Pp,Qp = P_Q_and_derivs(n)

    H = np.zeros((M,M), dtype=DT)

    diag_l_eq_k = [idx[v] for v in ["AA_XX","AB_XX","BB_XX","AA_YY","AB_YY","BB_YY"]]
    diag_j_eq_i = [idx[v] for v in ["AA_XX","BB_XX","AA_XY","BB_XY","AA_YY","BB_YY"]]

    for p in range(M):
        for q in range(M):
            term = DT("0.0")

            # T1_pq
            term += dg_p[p,q] + dg_p[q,p]
            term += np.dot(n, dg_pq[:,p,q])

            # T2_pq
            for r in diag_l_eq_k:
                term += Pp[r,p]*dg_p[r,q] + Pp[r,q]*dg_p[r,p] + P[r]*dg_pq[r,p,q]

            # T3_pq
            for r in diag_j_eq_i:
                term += Qp[r,p]*dg_p[r,q] + Qp[r,q]*dg_p[r,p] + Q[r]*dg_pq[r,p,q]

            H[p,q] = DT("0.5") * term

    return H

def fro_norm(M):
    M = asDT(M)
    return np.sqrt(np.sum(M * M, dtype=DT))


def H_fd(n0, h):
    n0 = asDT(n0)
    h = DT(h)
    if np.min(n0) <= h:
        raise ValueError("h too large: n0-h must stay positive for all components.")

    H = np.zeros((M,M), dtype=DT)
    G0 = G_ex(n0)

    for p in range(M):
        e = np.zeros(M, dtype=DT); e[p]=DT("1.0")
        H[p,p] = (G_ex(n0+h*e) - DT("2.0") * G0 + G_ex(n0-h*e)) / (h*h)

    for p in range(M):
        ep = np.zeros(M, dtype=DT); ep[p]=DT("1.0")
        for q in range(p+1,M):
            eq = np.zeros(M, dtype=DT); eq[q]=DT("1.0")
            Gpp = G_ex(n0+h*ep+h*eq)
            Gpm = G_ex(n0+h*ep-h*eq)
            Gmp = G_ex(n0-h*ep+h*eq)
            Gmm = G_ex(n0-h*ep-h*eq)
            val = (Gpp - Gpm - Gmp + Gmm) / (DT("4.0") * h*h)
            H[p,q]=val; H[q,p]=val

    return H

if __name__ == "__main__":
    n0 = np.array([DT("1.0"), DT("0.8"), DT("0.6"),
                   DT("0.7"), DT("0.4"), DT("0.3"),
                   DT("0.9"), DT("0.7"), DT("0.5")], dtype=DT)

    print("Vars:")
    for i,v in enumerate(vars_): print(i, v)
    print("\nG_ex(n0) =", G_ex(n0))

    Ha = H_ex_analytic(n0)
    print("||H_analytic||_F =", float(fro_norm(Ha)))
    print("analytic symmetry err =", float(fro_norm(Ha-Ha.T)/max(DT("1.0"), fro_norm(Ha))))

    for h in [DT("1e-2"), DT("1e-3"), DT("1e-4"), DT("1e-5"), DT("1e-6")]:
        if np.min(n0) <= h: 
            print("skip h",h); 
            continue
        Hfd = H_fd(n0,h)
        rel = fro_norm(Hfd-Ha)/max(DT("1.0"), fro_norm(Ha))
        print(f"h={h:g}  rel_err={rel:.3e}")

Vars:
0 AA_XX
1 AB_XX
2 BB_XX
3 AA_XY
4 AB_XY
5 BB_XY
6 AA_YY
7 AB_YY
8 BB_YY

G_ex(n0) = 1985.4331359234640786
||H_analytic||_F = 1004.3076818041588
analytic symmetry err = 0.0
h=0.01  rel_err=3.087e-05
h=0.001  rel_err=3.087e-07
h=0.0001  rel_err=3.070e-09
h=1e-05  rel_err=6.147e-09
h=1e-06  rel_err=5.835e-07
